## 导入模型进行测试

In [2]:
# INT8-only validation (robust)
from pathlib import Path
import numpy as np
import tensorflow as tf

val_root = Path("../dataset_wlx/validation")
tflite_path = Path("models/v2.int8.tflite")
thumb_size = (64, 64)
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not val_root.exists():
    raise FileNotFoundError(f"Validation path not found: {val_root.resolve()}")
if not tflite_path.exists():
    raise FileNotFoundError(f"TFLite file not found: {tflite_path.resolve()}")

class_names = sorted([d.name for d in val_root.iterdir() if d.is_dir()])
class_to_idx = {c: i for i, c in enumerate(class_names)}

samples = []
for cname in class_names:
    for p in sorted((val_root / cname).iterdir()):
        if p.suffix.lower() in exts:
            samples.append((p, class_to_idx[cname]))

if not samples:
    raise RuntimeError(f"No validation images found under: {val_root.resolve()}")

interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
in_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
in_scale, in_zero = in_d["quantization"]
out_scale, out_zero = out_d["quantization"]

if in_d["dtype"] != np.int8:
    raise TypeError(f"Expected int8 input tensor, got: {in_d['dtype']}")
if in_scale == 0:
    raise ValueError("Invalid input quantization scale (0).")

print("[INFO] classes:", class_names)
print("[INFO] total samples:", len(samples))
print("[INFO] tflite:", tflite_path)
print("[INFO] tflite input shape/dtype:", in_d["shape"], in_d["dtype"])
print("[INFO] tflite input quant:", in_d["quantization"])
print("[INFO] tflite output quant:", out_d["quantization"])

y_true = []
y_pred_int8 = []

for p, true_idx in samples:
    img = tf.keras.utils.load_img(p, color_mode="grayscale", target_size=thumb_size)
    x = tf.keras.utils.img_to_array(img).astype(np.float32) / 255.0
    x = x[np.newaxis, ...]

    x_q = np.clip(np.round(x / in_scale + in_zero), -128, 127).astype(np.int8)
    interp.set_tensor(in_d["index"], x_q)
    interp.invoke()
    y_q = interp.get_tensor(out_d["index"])[0]
    y_i = (y_q.astype(np.float32) - out_zero) * out_scale
    pi = int(np.argmax(y_i))

    y_true.append(true_idx)
    y_pred_int8.append(pi)

y_true = np.array(y_true, dtype=np.int32)
y_pred_int8 = np.array(y_pred_int8, dtype=np.int32)

acc_i = float((y_true == y_pred_int8).mean())
print(f"\n[RESULT] int8 acc = {acc_i:.4f}")

print("\n[PER-CLASS int8]")
for cname, idx in class_to_idx.items():
    m = (y_true == idx)
    ai = float((y_pred_int8[m] == idx).mean()) if m.any() else 0.0
    print(f"{cname:<10} n={m.sum():<4} int8={ai:.4f}")

# confusion matrix (rows=true, cols=pred)
num_classes = len(class_names)
cm = np.zeros((num_classes, num_classes), dtype=np.int32)
for t, p in zip(y_true, y_pred_int8):
    cm[t, p] += 1

print("\n[CONFUSION MATRIX] rows=true, cols=pred")
print("labels:", class_names)
print(cm)



[INFO] classes: ['paper', 'rock', 'scissors']
[INFO] total samples: 850
[INFO] tflite: models/v2.int8.tflite
[INFO] tflite input shape/dtype: [ 1 64 64  1] <class 'numpy.int8'>
[INFO] tflite input quant: (0.003921568859368563, -128)
[INFO] tflite output quant: (0.00390625, -128)

[RESULT] int8 acc = 0.9329

[PER-CLASS int8]
paper      n=295  int8=0.9322
rock       n=271  int8=0.9557
scissors   n=284  int8=0.9120

[CONFUSION MATRIX] rows=true, cols=pred
labels: ['paper', 'rock', 'scissors']
[[275   4  16]
 [ 10 259   2]
 [ 25   0 259]]
